# Building Qini Curve

The Qini curve is a popular evaluation metric for uplift modeling. It visualizes the cumulative incremental lift across the population sorted by predicted uplift scores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyuplift.datasets import make_linear_regression
from pyuplift.model_selection import train_test_split
from pyuplift.variable_selection import Dummy

# Generate synthetic uplift dataset
df = make_linear_regression(10000, random_state=42)
X, y, t = df.drop(['y', 't'], axis=1).values, df['y'].values, df['t'].values

# Split into train and test sets
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(X, y, t, random_state=42)

# Train a dummy uplift model
model = Dummy()
model.fit(X_train, y_train, t_train)

# Predict uplift scores for the test set
y_pred = model.predict(X_test)

In [ ]:
# Calculate and plot the Qini curve
order = np.argsort(y_pred)[::-1]
y_test_sorted = y_test[order]
t_test_sorted = t_test[order]

n = len(y_test_sorted)
proportions = np.arange(n + 1) / n

cum_treated = np.zeros(n + 1)
cum_control = np.zeros(n + 1)

for i in range(1, n + 1):
    treated = y_test_sorted[:i][t_test_sorted[:i] == 1]
    control = y_test_sorted[:i][t_test_sorted[:i] == 0]
    cum_treated[i] = cum_treated[i-1] + treated.sum()
    cum_control[i] = cum_control[i-1] + control.sum()

qini_curve = (cum_treated - cum_control) / n

plt.figure(figsize=(8, 6))
plt.plot(proportions, qini_curve, label='Qini Curve', linewidth=2)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Proportion of Population', fontsize=12)
plt.ylabel('Cumulative Uplift', fontsize=12)
plt.title('Qini Curve for Uplift Model', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

## Conclusion

A higher Qini curve indicates better model performance in identifying high-uplift individuals. The area under the Qini curve can be used to quantitatively compare different uplift models.